# 01 – Download & EDA
Download SmartDoc-QA, verify SD7K structure, run exploratory analysis.

In [ ]:
# ─── Install dependencies ───────────────────────────────────────────────────
!pip -q install segmentation-models-pytorch albumentations jiwer pytesseract \
  scikit-image pytorch-msssim gradio easyocr pyyaml tqdm matplotlib seaborn joblib
# On Colab: also install tesseract binary
import subprocess, sys
if 'google.colab' in sys.modules:
    subprocess.run(['apt-get', '-qq', 'install', 'tesseract-ocr'], check=True)

In [ ]:
# ─── Mount Google Drive (Colab) ─────────────────────────────────────────────
import sys
COLAB = 'google.colab' in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DocRescue'
else:
    PROJECT_ROOT = '.'  # local: repo root
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ─── Download SmartDoc-QA from Zenodo ───────────────────────────────────────
import os, zipfile
SMARTDOC_URL = 'https://zenodo.org/records/5293201/files/SmartDocQA.zip'
SMARTDOC_DIR = os.path.join(PROJECT_ROOT, 'data', 'smartdoc_qa')

os.makedirs(SMARTDOC_DIR, exist_ok=True)

zip_path = os.path.join(PROJECT_ROOT, 'data', 'SmartDocQA.zip')
if not os.path.exists(zip_path):
    print('Downloading SmartDoc-QA (~500 MB) …')
    !wget -q --show-progress '{SMARTDOC_URL}' -O '{zip_path}'
    print('Extracting …')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(SMARTDOC_DIR)
    print('Done.')
else:
    print('SmartDoc-QA already downloaded.')

In [ ]:
# ─── Verify SD7K local structure ────────────────────────────────────────────
# SD7K is already on your machine. Edit this path:
SD7K_RAW = os.path.join(PROJECT_ROOT, 'data', 'sd7k_raw')

for split in ['train', 'test']:
    input_dir = os.path.join(SD7K_RAW, split, 'input')
    target_dir = os.path.join(SD7K_RAW, split, 'target')
    n_i = len(os.listdir(input_dir))  if os.path.isdir(input_dir)  else 0
    n_t = len(os.listdir(target_dir)) if os.path.isdir(target_dir) else 0
    print(f'{split}: {n_i} input, {n_t} target')

In [ ]:
# ─── EDA: Visualise SD7K pairs ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import random
from PIL import Image

input_files = sorted(os.listdir(os.path.join(SD7K_RAW, 'train', 'input')))
samples = random.sample(input_files, min(6, len(input_files)))

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for col, fn in enumerate(samples):
    s = Image.open(os.path.join(SD7K_RAW, 'train', 'input', fn)).resize((256, 256))
    c = Image.open(os.path.join(SD7K_RAW, 'train', 'target', fn)).resize((256, 256))
    axes[0, col].imshow(s); axes[0, col].set_title(fn[:15], fontsize=7); axes[0, col].axis('off')
    axes[1, col].imshow(c); axes[1, col].axis('off')
axes[0, 0].set_ylabel('Input (shadow)', fontsize=9)
axes[1, 0].set_ylabel('Target (shadow-free)', fontsize=9)
plt.suptitle('SD7K Training Pairs (random sample)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'sd7k_sample_pairs.png'), dpi=100)
plt.show()

In [ ]:
# ─── EDA: Image size distribution ───────────────────────────────────────────
import numpy as np
import seaborn as sns

widths, heights = [], []
for fn in input_files[:500]:  # sample 500
    with Image.open(os.path.join(SD7K_RAW, 'train', 'input', fn)) as img:
        w, h = img.size
        widths.append(w); heights.append(h)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(widths, ax=axes[0], bins=20, color='steelblue')
axes[0].set_title('Width distribution'); axes[0].set_xlabel('pixels')
sns.histplot(heights, ax=axes[1], bins=20, color='coral')
axes[1].set_title('Height distribution'); axes[1].set_xlabel('pixels')
plt.tight_layout(); plt.show()
print(f'Median size: {int(np.median(widths))}×{int(np.median(heights))}')

In [ ]:
# ─── EDA: Pixel statistics (brightness before/after) ────────────────────────
means_input, means_target = [], []
for fn in random.sample(input_files, min(200, len(input_files))):
    s = np.array(Image.open(os.path.join(SD7K_RAW, 'train', 'input', fn)).convert('L'))
    c = np.array(Image.open(os.path.join(SD7K_RAW, 'train', 'target', fn)).convert('L'))
    means_input.append(s.mean())
    means_target.append(c.mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(means_input,  bins=30, alpha=0.6, label='Input (shadow)', color='navy')
ax.hist(means_target, bins=30, alpha=0.6, label='Target (shadow-free)', color='orange')
ax.set_xlabel('Mean pixel intensity'); ax.set_ylabel('Count')
ax.set_title('Brightness distribution: input vs target')
ax.legend(); plt.tight_layout(); plt.show()